# Pipeline Step 2: Temporal Feature Engineering (EMA)

## Inputs
- **Preprocessed SofaScore Dataset**: `sofa_preprocessed.csv` (from Notebook 1)
- **Preprocessed Shot Events Dataset**: `shots_preprocessed.csv` (from Notebook 1)

## Outputs
- **EMA Pre-State Table**: `ema_pre_table.csv` (calculated player pre-match Exponential Moving Average form states)
- **Joined and Encoded Dataset**: `final_total_input.xlsx` (final shot-by-shot dataset containing encoded shot features and pre-match player/keeper forms)


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

print("Libraries loaded successfully.")


Libraries loaded successfully.


### Configuration Parameters
Define mathematical constants for Exponential Moving Average (EMA) smoothing and windowing.

In [2]:
HALF_LIFE_VAL = 56        # decay half-life in days (~8 weeks)
CLAMP_MAX_GAP_DAYS = 14   # compress off-season gaps to at most 14 days

print(f"EMA Config: Half-Life={HALF_LIFE_VAL} days | Clamp Gap={CLAMP_MAX_GAP_DAYS} days")
import unicodedata

def normalize_txt(s):
    if pd.isna(s):
        return ""
    val = str(s).strip().lower()
    nfkd_form = unicodedata.normalize('NFKD', val)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

EMA Config: Half-Life=56 days | Clamp Gap=14 days


### Exponential Moving Average (EMA) Builder
Computes Exponential Moving Average (EMA) for player statistics match-by-match. 
To prevent data leakage, it tracks pre-match form (meaning state strictly *before* the match date, updating it *after* recording).

In [3]:
def build_ema_table(
    df: pd.DataFrame,
    name_col: str,
    date_col: str = "Date",
    features: Optional[List[str]] = None,
    half_life_mode: str = "days",
    half_life_val: float = 56.0,
    clamp_days: int = 14,
    prefix: str = "pre__"
) -> pd.DataFrame:
    """
    Calculates the player performance EMA pre-match form vector chronologically.
    Excludes target match date stats from current state to prevent leakage.
    """
    df = df.copy()
    df = df.sort_values([name_col, date_col]).reset_index(drop=True)

    # Auto-detect numerical features to track if not specified
    if features is None:
        candidates = []
        for c in df.columns:
            if c in [name_col, date_col]:
                continue
            if np.issubdtype(df[c].dtype, np.number):
                candidates.append(c)
        features = candidates
        print(f"Detected {len(features)} numeric performance features to scale.")

    decay = 0.5 ** (1.0 / half_life_val)
    out_records = []

    for ent, g in df.groupby(name_col, sort=False):
        g = g.sort_values(date_col)

        # Initialize player pre-state to zero vectors
        init_vals = {f: 0.0 for f in features}
        prev_state = init_vals.copy()
        prev_date = None

        # Seed a record one day prior to first appearance to initialize histories
        first_date = g[date_col].iloc[0]
        seed_date = first_date - pd.Timedelta(days=1)
        seed = {name_col: ent, date_col: seed_date}
        for f in features:
            seed[f"{prefix}{f}"] = prev_state[f]
        out_records.append(seed)

        for _, row in g.iterrows():
            cur_date = row[date_col]
            # Record pre-match form state (strictly prior performance)
            rec = {name_col: ent, date_col: cur_date}
            for f in features:
                rec[f"{prefix}{f}"] = prev_state[f]
            out_records.append(rec)

            # Compute elapsed days delta
            if half_life_mode == "matches":
                delta = 1.0
            else:
                if prev_date is None or pd.isna(prev_date) or pd.isna(cur_date):
                    delta = 1.0
                else:
                    gap = max((cur_date - prev_date).days, 1)
                    delta = float(min(gap, clamp_days))  # Compress offseason gaps

            # Update EMA state
            w = decay ** delta
            for f in features:
                x_t = row[f] if pd.notna(row[f]) else prev_state[f]
                prev_state[f] = w * prev_state[f] + (1.0 - w) * x_t

            prev_date = cur_date

    ema_df = pd.DataFrame(out_records)
    return ema_df


### Loading SofaScore Data and Running EMA Pipeline
Load preprocessed player performance data, calculate EMA table, and save.

In [4]:
print("Loading preprocessed SofaScore dataset...")
sofa_df = pd.read_csv("sofa_preprocessed.csv")
sofa_df["Date"] = pd.to_datetime(sofa_df["Date"])
sofa_df["Player"] = sofa_df["Player"].apply(normalize_txt)

# Identify features for EMA (excluding metadata and positioning)
exclude_cols = {"Player", "Date", "Team", "Score", "Home/Away", "Player_ID"}
ema_features = [c for c in sofa_df.columns if c not in exclude_cols and np.issubdtype(sofa_df[c].dtype, np.number)]

print(f"Calculating player EMA table for {len(ema_features)} features...")
ema_table = build_ema_table(
    df=sofa_df,
    name_col="Player",
    date_col="Date",
    features=ema_features,
    half_life_mode="days",
    half_life_val=HALF_LIFE_VAL,
    clamp_days=CLAMP_MAX_GAP_DAYS,
    prefix="pre__"
)

print(f"Player EMA table shape: {ema_table.shape}")
ema_table.to_csv("ema_pre_table.csv", index=False)
print("Saved player EMA table to 'ema_pre_table.csv'.")


Loading preprocessed SofaScore dataset...
Calculating player EMA table for 37 features...
Player EMA table shape: (15409, 39)
Saved player EMA table to 'ema_pre_table.csv'.


### Shot Events Encoding and Merging
Define helper functions to one-hot encode shot categorical features, and execute asof joins to merge player/keeper EMA vectors into shot records.

In [5]:
class ShotFeatureSpec:
    def __init__(self, feature_cols, categorical_cols, numeric_cols):
        self.feature_cols = feature_cols
        self.categorical_cols = categorical_cols
        self.numeric_cols = numeric_cols

def fit_transform_shot_features(df: pd.DataFrame, spec: ShotFeatureSpec) -> Tuple[pd.DataFrame, ColumnTransformer, List[str]]:
    """
    One-hot encodes categorical shot columns and keeps numeric columns.
    """
    ct = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), spec.categorical_cols),
            ("num", "passthrough", spec.numeric_cols)
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )
    X = ct.fit_transform(df[spec.feature_cols])
    shot_cols = list(ct.get_feature_names_out())
    X_df = pd.DataFrame(X, index=df.index, columns=[f"shots_enc__shots__{c}" for c in shot_cols]).astype(np.float32)
    return X_df, ct, [f"shots_enc__shots__{c}" for c in shot_cols]

def assemble_single_table(
    shots_df: pd.DataFrame,
    shot_feature_spec: ShotFeatureSpec,
    shooter_ema: pd.DataFrame,
    keeper_ema: pd.DataFrame,
    shooter_name_col: str = "shooter",
    keeper_name_col: str = "goalie",
    date_col: str = "Date",
    target_col: str = "Goal",
    prefix_sh: str = "sh__",
    prefix_gk: str = "gk__",
) -> pd.DataFrame:
    """
    Performs asof joins to merge the shooter and goalkeeper EMA forms into the shot events table.
    """
    df = shots_df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    shooter_ema[date_col] = pd.to_datetime(shooter_ema[date_col])
    keeper_ema[date_col]  = pd.to_datetime(keeper_ema[date_col])

    # Sort chronologically for asof merge
    df = df.sort_values(date_col)
    shooter_ema = shooter_ema.sort_values(date_col)
    keeper_ema  = keeper_ema.sort_values(date_col)

    # Perform asof merge (capturing state strictly BEFORE the match/shot date)
    sh_merge = pd.merge_asof(
        df,
        shooter_ema.rename(columns={"Player": shooter_name_col}),
        left_on=date_col,
        right_on=date_col,
        by=shooter_name_col,
        direction="backward",
        allow_exact_matches=False
    )

    gk_merge = pd.merge_asof(
        df,
        keeper_ema.rename(columns={"Player": keeper_name_col}),
        left_on=date_col,
        right_on=date_col,
        by=keeper_name_col,
        direction="backward",
        allow_exact_matches=False
    )

    # Extract features for blocks
    sh_feat = [c for c in shooter_ema.columns if c not in ("Player", date_col)]
    gk_feat = [c for c in keeper_ema.columns if c not in ("Player", date_col)]

    sh_block = sh_merge[sh_feat].copy()
    sh_block.columns = [prefix_sh + c for c in sh_block.columns]
    gk_block = gk_merge[gk_feat].copy()
    gk_block.columns = [prefix_gk + c for c in gk_block.columns]

    # Encode physical shot features
    X_shot_enc, shot_enc, shot_cols = fit_transform_shot_features(df, shot_feature_spec)

    # Save encoder artifact
    os.makedirs("artifacts", exist_ok=True)
    joblib.dump(shot_enc, "artifacts/shot_encoder.joblib")

    # Format metadata columns to match final_total_input.xlsx structure
    df_meta = df[[date_col, shooter_name_col, 'Team', keeper_name_col, 'goalie team', target_col]].copy()
    df_meta.columns = ['date', 'shooter', 'shooter_team', 'goalie', 'goalie_team', 'is_Goal']

    # Concat all features together
    out = pd.concat([
        df_meta.reset_index(drop=True),
        X_shot_enc.reset_index(drop=True),
        sh_block.reset_index(drop=True),
        gk_block.reset_index(drop=True),
    ], axis=1)

    # Fill NaN values (players/keepers with no prior history) with 0
    out = out.fillna(0)

    return out


### Loading Preprocessed Shots & Assembling Dataset
Assemble the final merged dataset and write it as `final_total_input.xlsx`.

In [6]:

print("Loading preprocessed shot events...")
shots_df = pd.read_csv("shots_preprocessed.csv")
shots_df["Date"] = pd.to_datetime(shots_df["Date"])
shots_df["shooter"] = shots_df["shooter"].apply(normalize_txt)
shots_df["goalie"] = shots_df["goalie"].apply(normalize_txt)

# Categorical vs numerical shot features
SHOT_CATEGORICAL = ["Position", "FieldWidth", "FieldLocation", "ShotPatternOfPlay", "ShotBodyPart", "Pressure"]
SHOT_NUMERIC = ["EventX", "EventY", "GKx", "GKy", "homeCurrentScore", "awayCurrentScore", "PossNum", "PlyrsBtwn", "sequenceSeconds", "possessionSeconds"]

spec = ShotFeatureSpec(
    feature_cols=SHOT_CATEGORICAL + SHOT_NUMERIC,
    categorical_cols=SHOT_CATEGORICAL,
    numeric_cols=SHOT_NUMERIC
)

print("Assembling combined dataset with asof joins...")
final_total_input = assemble_single_table(
    shots_df=shots_df,
    shot_feature_spec=spec,
    shooter_ema=ema_table.copy(),
    keeper_ema=ema_table.copy(),
    shooter_name_col="shooter",
    keeper_name_col="goalie",
    date_col="Date",
    target_col="Goal",
    prefix_sh="sh__",
    prefix_gk="gk__"
)

print(f"Final dataset shape: {final_total_input.shape}")
print(f"Total columns: {len(final_total_input.columns)}")

# Save to Excel
final_total_input.to_excel("final_total_input.xlsx", index=False)
print("Saved final merged dataset to 'final_total_input.xlsx'.")


Loading preprocessed shot events...
Assembling combined dataset with asof joins...
Final dataset shape: (11571, 129)
Total columns: 129
Saved final merged dataset to 'final_total_input.xlsx'.
